In [9]:
import json
import re
from datetime import datetime, timezone
import pandas as pd


def process_bronze_to_silver(input_json_path: str):
    # ---------------------------------------------------------
    # BƯỚC 1: Nạp Bronze Data & Thiết lập Metadata Batch
    # ---------------------------------------------------------
    with open(input_json_path, "r", encoding="utf-8") as f:
        raw_records = json.load(f)

    batch_id = f"BATCH_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
    ingestion_timestamp = (
        datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    )
    source_file_name = input_json_path.split("/")[-1]

    silver_records = []
    quarantine_records = []

    # Danh mục kênh chuẩn để kiểm tra hợp lệ
    VALID_CHANNELS = {
        "organic_search",
        "paid_search",
        "social_media",
        "email_campaign",
        "referral",
        "direct",
    }

    # ---------------------------------------------------------
    # BƯỚC 2: Kiểm tra dữ liệu từng dòng (Validation & Profiling)
    # ---------------------------------------------------------
    for idx, row in enumerate(raw_records):
        errors = []

        # 1. Kiểm tra Ngày (date)
        raw_date = row.get("date")
        parsed_date = None
        if not raw_date:
            errors.append(("MISSING_DATE", "Trường date bị thiếu hoặc rỗng"))
        else:
            try:
                # Kiểm tra định dạng ngày chuẩn YYYY-MM-DD
                dt = datetime.strptime(str(raw_date).strip(), "%Y-%m-%d")
                # Kiểm tra năm hợp lệ (tránh các năm lỗi giả lập hoặc lỗi định dạng)
                if dt.year < 2000 or dt.year > 2026:
                    errors.append(
                        (
                            "INVALID_DATE",
                            f"Năm ngoài khoảng báo cáo hợp lệ: {raw_date}",
                        )
                    )
                else:
                    parsed_date = dt.strftime("%Y-%m-%d")
            except ValueError:
                errors.append(
                    ("INVALID_DATE", f"Ngày không đúng định dạng: {raw_date}")
                )

        # 2. Kiểm tra Sessions
        raw_sessions = row.get("sessions")
        parsed_sessions = None
        if raw_sessions is None or str(raw_sessions).strip() in [
            "",
            "null",
            "N/A",
            "nan",
        ]:
            errors.append(
                ("MISSING_SESSIONS", "Sessions bị null hoặc thiếu dữ liệu")
            )
        else:
            try:
                s_val = float(raw_sessions)
                if not s_val.is_integer() or s_val < 0:
                    errors.append(
                        (
                            "INVALID_SESSIONS",
                            f"Sessions phải là số nguyên >= 0: {raw_sessions}",
                        )
                    )
                elif s_val > 10_000_000:  # Giá trị bất thường / outlier
                    errors.append(
                        (
                            "OUTLIER_SESSIONS",
                            f"Sessions vượt ngưỡng cho phép: {raw_sessions}",
                        )
                    )
                else:
                    parsed_sessions = int(s_val)
            except (ValueError, TypeError):
                errors.append(
                    (
                        "INVALID_SESSIONS",
                        f"Sessions không thể ép kiểu sang số: {raw_sessions}",
                    )
                )

        # 3. Kiểm tra Unique Visitors
        raw_uv = row.get("unique_visitors")
        parsed_uv = None
        if raw_uv is None or str(raw_uv).strip() in [
            "",
            "null",
            "N/A",
            "nan",
            "unknown",
        ]:
            errors.append(
                (
                    "MISSING_UNIQUE_VISITORS",
                    "unique_visitors bị null, chuỗi không xác định hoặc thiếu",
                )
            )
        else:
            try:
                uv_val = float(raw_uv)
                if not uv_val.is_integer() or uv_val < 0:
                    errors.append(
                        (
                            "INVALID_UNIQUE_VISITORS",
                            f"unique_visitors phải là số nguyên >= 0: {raw_uv}",
                        )
                    )
                elif parsed_sessions is not None and uv_val > parsed_sessions:
                    errors.append(
                        (
                            "INVALID_GRAIN_LOGIC",
                            f"unique_visitors ({uv_val}) không thể lớn hơn sessions ({parsed_sessions})",
                        )
                    )
                else:
                    parsed_uv = int(uv_val)
            except (ValueError, TypeError):
                errors.append(
                    (
                        "INVALID_UNIQUE_VISITORS",
                        f"unique_visitors không thể ép kiểu sang số: {raw_uv}",
                    )
                )

        # 4. Kiểm tra Traffic Source & Chuẩn hóa Kênh
        raw_ts = row.get("traffic_source")
        parsed_channel = None
        parsed_variant = None
        clean_ts = str(raw_ts).strip() if raw_ts is not None else ""

        if not clean_ts or clean_ts.upper() in ["N/A", "NULL", "NONE"]:
            errors.append(
                (
                    "MISSING_TRAFFIC_SOURCE",
                    f"Traffic source bị trống hoặc không hợp lệ: '{raw_ts}'",
                )
            )
        else:
            # Tách channel và variant (ví dụ: 'paid_search Variant 0002')
            match = re.match(r"^(.*?)(?:\s+(Variant\s+\d+))?$", clean_ts)
            if match:
                parsed_channel = match.group(1).strip()
                parsed_variant = match.group(2) if match.group(2) else None

            if parsed_channel not in VALID_CHANNELS:
                errors.append(
                    (
                        "UNMAPPED_CATEGORY",
                        f"Kênh traffic chưa được ánh xạ trong danh mục: '{parsed_channel}'",
                    )
                )

        # ---------------------------------------------------------
        # BƯỚC 3: Phân luồng Silver và Quarantine
        # ---------------------------------------------------------
        if errors:
            # Lưu toàn bộ lỗi phát hiện được vào quarantine
            for err_code, err_msg in errors:
                quarantine_records.append(
                    {
                        "source_file": source_file_name,
                        "source_record_idx": idx,
                        "error_code": err_code,
                        "error_message": err_msg,
                        "raw_record": json.dumps(row, ensure_ascii=False),
                        "batch_id": batch_id,
                        "ingestion_timestamp": ingestion_timestamp,
                    }
                )
        else:
            # Chuẩn hóa bản ghi Silver
            silver_records.append(
                {
                    "source_record_idx": idx,
                    "date": parsed_date,
                    "reporting_period": parsed_date[:7],  # YYYY-MM
                    "traffic_channel": parsed_channel,
                    "traffic_variant": parsed_variant,
                    "sessions": parsed_sessions,
                    "unique_visitors": parsed_uv,
                    "page_views": int(row.get("page_views", 0)),
                    "bounce_rate": round(float(row.get("bounce_rate", 0.0)), 6),
                    "avg_session_duration_sec": round(
                        float(row.get("avg_session_duration_sec", 0.0)), 2
                    ),
                    "source_file": source_file_name,
                    "ingestion_timestamp": ingestion_timestamp,
                    "batch_id": batch_id,
                }
            )

    # ---------------------------------------------------------
    # BƯỚC 4: Kiểm tra Deduplication trên tập Silver
    # ---------------------------------------------------------
    df_silver = pd.DataFrame(silver_records)
    # Khóa nghiệp vụ: date + traffic_channel
    dup_mask = df_silver.duplicated(
        subset=["date", "traffic_channel"], keep="first"
    )
    if dup_mask.any():
        dups = df_silver[dup_mask]
        for _, dup_row in dups.iterrows():
            quarantine_records.append(
                {
                    "source_file": source_file_name,
                    "source_record_idx": dup_row["source_record_idx"],
                    "error_code": "DUPLICATE_BUSINESS_KEY",
                    "error_message": f"Trùng khóa nghiệp vụ: {dup_row['date']} - {dup_row['traffic_channel']}",
                    "raw_record": json.dumps(
                        dup_row.to_dict(), ensure_ascii=False
                    ),
                    "batch_id": batch_id,
                    "ingestion_timestamp": ingestion_timestamp,
                }
            )
        df_silver = df_silver[~dup_mask].copy()

    df_quarantine = pd.DataFrame(quarantine_records)

    # ---------------------------------------------------------
    # BƯỚC 5: Xuất dữ liệu ra file
    # ---------------------------------------------------------
    df_silver.to_json(
        "silver_traffic_data.json",
        orient="records",
        indent=2,
        force_ascii=False,
    )
    df_quarantine.to_json(
        "quarantine_traffic_data.json",
        orient="records",
        indent=2,
        force_ascii=False,
    )

    # ---------------------------------------------------------
    # BƯỚC 6: Báo cáo đối soát & Chất lượng dữ liệu (Reconciliation)
    # ---------------------------------------------------------
    total_bronze = len(raw_records)
    total_silver = len(df_silver)
    # Số bản ghi thô bị đưa vào quarantine (tính theo distinct source_record_idx)
    total_quarantine_records = (
        df_quarantine["source_record_idx"].nunique()
        if not df_quarantine.empty
        else 0
    )

    print("=" * 60)
    print("BÁO CÁO ĐỐI SOÁT & NGHIỆM THU DỮ LIỆU (RECONCILIATION REPORT)")
    print("=" * 60)
    print(f"Mã Batch             : {batch_id}")
    print(f"Tổng số bản ghi Bronze: {total_bronze}")
    print(
        f"Số bản ghi Silver     : {total_silver} ({total_silver / total_bronze * 100:.2f}%)"
    )
    print(
        f"Số bản ghi Quarantine : {total_quarantine_records} ({total_quarantine_records / total_bronze * 100:.2f}%)"
    )

    # Đối soát tổng số dòng
    is_balanced = total_bronze == (total_silver + total_quarantine_records)
    print(f"Kiểm tra cân bằng dòng: {'KHỚP [PASS]' if is_balanced else 'LỆCH [FAIL]'}")

    # Chỉ số kiểm soát Silver
    print("-" * 60)
    print(f"Tổng Sessions Silver  : {df_silver['sessions'].sum():,}")
    print(
        f"Tổng Visitors Silver  : {df_silver['unique_visitors'].sum():,}"
    )
    print(f"Tổng Page Views Silver: {df_silver['page_views'].sum():,}")
    print(
        f"Tỷ lệ thoát TB        : {df_silver['bounce_rate'].mean():.6f}"
    )
    print(
        f"Thời lượng phiên TB   : {df_silver['avg_session_duration_sec'].mean():.2f}s"
    )
    print("=" * 60)


if __name__ == "__main__":
    process_bronze_to_silver("tf.json")

BÁO CÁO ĐỐI SOÁT & NGHIỆM THU DỮ LIỆU (RECONCILIATION REPORT)
Mã Batch             : BATCH_20260918_154321
Tổng số bản ghi Bronze: 1000
Số bản ghi Silver     : 990 (99.00%)
Số bản ghi Quarantine : 10 (1.00%)
Kiểm tra cân bằng dòng: KHỚP [PASS]
------------------------------------------------------------
Tổng Sessions Silver  : 24,960,983
Tổng Visitors Silver  : 18,945,934
Tổng Page Views Silver: 108,110,430
Tỷ lệ thoát TB        : 0.004479
Thời lượng phiên TB   : 206.40s
